# Mollier h,x-Diagram

This example demonstrates how to use `plot_mollier_hx` to create
psychrometric charts (Mollier h,x-diagrams) for visualising the state
of moist air.

All plots are **interactive** — hover over data points to see details.

## Empty diagram (coordinate grid only)

Pass `comfort_zone=False` to show only the iso-lines without a comfort zone.

In [ ]:
from pyedautils.plots import plot_mollier_hx
from IPython.display import HTML

html = plot_mollier_hx(comfort_zone=False)
HTML(html)

## With default comfort zone

Calling `plot_mollier_hx()` without arguments adds the default comfort zone
(T: 20–26 °C, φ: 30–65 %, x: 0–11.5 g/kg).

In [ ]:
html = plot_mollier_hx()
HTML(html)

## Diagram with measured data

Pass a DataFrame with columns `[timestamp, humidity, temperature]`
(humidity in %, temperature in °C) to overlay data points.
Points are automatically colour-coded by season.

In [ ]:
import pandas as pd
from importlib import resources

data_path = resources.files("pyedautils") / "data" / "mollier_sample.csv"
df = pd.read_csv(data_path)
df.head()

In [ ]:
html = plot_mollier_hx(data=df)
HTML(html)

## Customisation

You can adjust the pressure (e.g. for higher altitude) and change the
comfort zone.

In [ ]:
html = plot_mollier_hx(
    data=df,
    pressure=95000.0,  # ~500 m altitude
    comfort_zone={
        "temperature": (18, 24),
        "rel_humidity": (0.20, 0.70),
        "abs_humidity": (0, 0.012),
    },
)
HTML(html)

## Custom axis ranges

Use `domain_x` and `domain_y` to zoom into a specific region of the diagram.
`domain_x` controls the absolute humidity range (kg/kg) and `domain_y`
controls the y-coordinate range (≈ temperature at x=0).

In [ ]:
html = plot_mollier_hx(
    data=df,
    domain_x=(0.002, 0.014),  # absolute humidity 2–14 g/kg
    domain_y=(10.0, 35.0),    # y-range ≈ 10–35 °C
)
HTML(html)

## Convention: classical vs. Glück

The y-axis transformation has two valid normalisations. `'classical'`
(default) follows Mollier 1923 / Recknagel — enthalpy per kg of dry
air, isotherms tilt slightly **up** with x. `'glueck'` follows the
Glück reference — enthalpy per kg of moist air, isotherms tilt slightly
**down**. Physical quantities (T, φ, ρ, h) are identical under either
convention; only the (x, y) parametrisation differs.

In [ ]:
html = plot_mollier_hx(data=df, convention='glueck')
HTML(html)

## Latest-record overlay

When `data` is supplied, the row with the newest timestamp is overlaid as a
black circle on top of the seasonal scatter — you've already seen it in the
diagrams above. The overlay is controlled by two parameters:

- `highlight_latest` (bool, default `True`) — show or hide the overlay.
- `highlight_color` (str | None, default `'black'`) — any CSS colour, or
  `None` to fall back to the row's season colour.

In [ ]:
# Disable the latest-row overlay
html = plot_mollier_hx(data=df, highlight_latest=False)
HTML(html)

In [ ]:
# Custom highlight colour
html = plot_mollier_hx(data=df, highlight_color='#e53935')
HTML(html)

## Process chain (psychrosim-style)

Use the `states=` argument to overlay a sequence of psychrometric state
points joined by arrows — a multi-process simulation in the spirit of
[psychrosim.com](https://www.psychrosim.com/). Build each state with the
`state(...)` factory in `pyedautils._mollier` and chain them through the
process functions (`heat`, `cool`, `humidify_adiabatic`, …).

In [ ]:
from pyedautils._mollier import (
    state, heat, humidify_adiabatic, heat_recovery,
)

p = 101325

# Winter: outdoor -5 °C / 80 % RH, extract 22 °C / 40 % RH, 1500 m³/h
s0 = state(t=-5, phi=0.80, p=p, volume_flow=1500)
extract = state(t=22, phi=0.40, p=p, volume_flow=1500)

s1, _ = heat_recovery(s0, extract, eps_sensible=0.75)
s2, _ = heat(s1, t_out=21)
s3, _ = humidify_adiabatic(s2, phi_out=0.45)

html = plot_mollier_hx(
    states=[s0, s1, s2, s3],
    labels=['Outdoor', 'After HR', 'After heater', 'After humidifier'],
    domain_x=(0.0, 0.012),
    domain_y=(-10, 30),
    comfort_zone=False,
)
HTML(html)